# CloudVision — اختبار CUDA على Kaggle T4

هذا الدفتر يختبر Real-ESRGAN فعلياً على GPU، ولا يكتفي بفحص تثبيت المكتبات. لا يرفع أي ملف إلى خدمة خارجية. شغّل الخلايا بالترتيب، وتأكد من اختيار GPU T4 من إعدادات الدفتر.

In [ ]:
import os, sys, subprocess, textwrap, pathlib

# احتفظ بإصدار PyTorch/CUDA المثبت مسبقاً في Kaggle؛ لا نعيد تثبيته.
def run(cmd):
    print('+', cmd)
    subprocess.run(cmd, shell=True, check=True)

import torch
print('Python:', sys.version.split()[0])
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('CUDA غير متاحة. من Notebook options اختر Accelerator = GPU T4 ثم أعد تشغيل الجلسة.')
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

# إصدارات متوافقة مع Python 3.13، دون استبدال Torch/TorchVision.
run(f'{sys.executable} -m pip install -q numpy==2.3.5 opencv-python-headless==4.11.0.86 addict future lmdb Pillow PyYAML requests scikit-image scipy tqdm yapf cython tensorboard')
run('git clone -q --depth 1 --branch v1.4.2 https://github.com/XPixelGroup/BasicSR.git /kaggle/working/BasicSR')
run('git clone -q --depth 1 --branch v0.3.0 https://github.com/xinntao/Real-ESRGAN.git /kaggle/working/Real-ESRGAN')

# إصلاح قارئ الإصدار القديم الذي يستخدم locals() داخل exec مع Python الحديث.
patch = '''from pathlib import Path\nfor p in [Path('/kaggle/working/BasicSR/setup.py'), Path('/kaggle/working/Real-ESRGAN/setup.py')]:\n    s = p.read_text()\n    old = \"\"\"def get_version():\\n    with open(version_file, 'r') as f:\\n        exec(compile(f.read(), version_file, 'exec'))\\n    return locals()['__version__']\"\"\"\n    new = \"\"\"def get_version():\\n    namespace = {}\\n    with open(version_file, 'r') as f:\\n        exec(compile(f.read(), version_file, 'exec'), namespace)\\n    return namespace['__version__']\"\"\"\n    if old in s:\n        p.write_text(s.replace(old, new))\n    elif 'namespace = {}' not in s:\n        raise RuntimeError(f'لم أجد دالة get_version المتوقعة في {p}')\n'''
exec(patch)
run(f'{sys.executable} -m pip install -q --no-build-isolation --no-deps /kaggle/working/BasicSR')
run(f'{sys.executable} -m pip install -q --no-build-isolation --no-deps /kaggle/working/Real-ESRGAN')
print('تم تثبيت المكتبات دون تغيير نسخة Torch.')

In [ ]:
import sys, types, pathlib, urllib.request, hashlib, time
import numpy as np
import torch

# توافق مع إصدارات TorchVision التي أزالت functional_tensor.
try:
    import torchvision.transforms.functional_tensor
except ModuleNotFoundError:
    import torchvision.transforms.functional as TF
    shim = types.ModuleType('torchvision.transforms.functional_tensor')
    shim.rgb_to_grayscale = TF.rgb_to_grayscale
    sys.modules['torchvision.transforms.functional_tensor'] = shim

from PIL import Image, ImageDraw
from basicsr.archs.rrdbnet_arch import RRDBNet
from realesrgan import RealESRGANer

assert torch.cuda.is_available(), 'CUDA غير متاحة'
device = torch.device('cuda:0')
print('GPU:', torch.cuda.get_device_name(0))

# صورة اختبار اصطناعية 128×128، حتى لا تحتاج إلى رفع صورة الآن.
img = Image.new('RGB', (128, 128), (35, 48, 65))
d = ImageDraw.Draw(img)
d.rectangle((16, 16, 111, 111), outline=(240, 190, 70), width=3)
d.ellipse((38, 38, 90, 90), fill=(70, 160, 210), outline=(245, 245, 245), width=2)
d.line((0, 127, 127, 0), fill=(230, 90, 100), width=3)
input_path = '/kaggle/working/cloudvision_input.png'
output_path = '/kaggle/working/cloudvision_t4_x4.png'
img.save(input_path)

weights = '/kaggle/working/RealESRGAN_x4plus.pth'
weights_url = 'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth'
if not os.path.exists(weights):
    print('Downloading official Real-ESRGAN x4plus weights...')
    urllib.request.urlretrieve(weights_url, weights)
sha = hashlib.sha256(open(weights, 'rb').read()).hexdigest()
print('Weights SHA256:', sha)

model = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64, num_block=23, num_grow_ch=32, scale=4)
upsampler = RealESRGANer(scale=4, model_path=weights, model=model, tile=256, tile_pad=10, pre_pad=0, half=True, device=device)
input_bgr = np.array(img)[:, :, ::-1].copy()
torch.cuda.reset_peak_memory_stats()
start = time.time()
output_bgr, _ = upsampler.enhance(input_bgr, outscale=4)
elapsed = time.time() - start
Image.fromarray(output_bgr[:, :, ::-1]).save(output_path)

print('\n===== CLOUDVISION T4 RESULT =====')
print('CUDA:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0))
print('Input:', img.size)
print('Output:', (output_bgr.shape[1], output_bgr.shape[0]))
print('Scale:', output_bgr.shape[1] // img.width, 'x')
print('Inference seconds:', round(elapsed, 3))
print('Peak VRAM GiB:', round(torch.cuda.max_memory_allocated() / 1024**3, 3))
print('Output file:', output_path)
assert output_bgr.shape[:2] == (512, 512), 'أبعاد الناتج غير متوقعة'
print('RESULT: PASS — Real-ESRGAN inference executed on CUDA GPU')
display(Image.open(input_path))
display(Image.open(output_path))

## ما الذي يثبته هذا الاختبار؟
- تشغيل النموذج وأوزانه الرسمية على CUDA/T4، وليس CPU.
- تكبير صورة اختبار من 128×128 إلى 512×512.
- تسجيل زمن الاستدلال واستهلاك VRAM.

هذا ليس اختبار فيديو بعد. بعد نجاحه نضيف مسار معالجة فيديو حقيقي مع تقسيم الإطارات وإعادة دمج الصوت، ثم نقيس السرعة على مقطع قصير.